In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from arch import arch_model
import plotly.graph_objects as go
from plotly.subplots import make_subplots

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [4]:

class VolatilityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class GatingDataset(Dataset):
    def __init__(self, X, y_real, y_garch):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y_real = torch.FloatTensor(y_real)
        self.y_garch = torch.FloatTensor(y_garch)
    def __len__(self): return len(self.y_real)
    def __getitem__(self, idx): return self.X[idx], self.y_real[idx], self.y_garch[idx]

class VolatilityLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=256, num_layers=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.batch_norm = nn.BatchNorm1d(hidden_size)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        normalized = self.batch_norm(last_output)
        fc1_out = self.relu(self.fc1(normalized))
        fc1_out = self.dropout(fc1_out)
        return self.fc2(fc1_out).squeeze(-1)

class FeatureGatingGINN(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.data_lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.data_bn = nn.BatchNorm1d(hidden_size)
        self.garch_proj = nn.Sequential(
            nn.Linear(1, hidden_size // 2), nn.ReLU(), nn.Linear(hidden_size // 2, hidden_size)
        )
        self.lambda_lstm = nn.LSTM(input_size, 32, 2, batch_first=True, bidirectional=True)
        self.lambda_bn = nn.BatchNorm1d(64)
        self.lambda_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1)
        )
    def forward(self, x, garch_signal):
        out, _ = self.data_lstm(x)
        data_feat = self.data_bn(out[:, -1, :])
        garch_feat = self.garch_proj(garch_signal)
        l_out, _ = self.lambda_lstm(x)
        l_feat = self.lambda_bn(l_out[:, -1, :])
        lambda_val = self.lambda_head(l_feat)
        combined = data_feat + lambda_val * garch_feat
        output = self.output_head(combined).squeeze(-1)
        return output, lambda_val.squeeze(-1)

In [41]:
imoex_df = pd.read_json("./data.json")
imoex_df.columns = ["BOARDID", "SECID", "TRADEDATE", "SHORTNAME", "NAME", "CLOSE", "OPEN", "HIGH", "LOW", "VALUE", "DURATION", "YIELD", "DECIMALS", "CAPITALIZATION", "CURRENCYID", "DIVISOR", "TRADINGSESSION", "VOLUME", "TRADE_SESSION_DATE", "RECALC_DATE"]
nasdq_df['LogReturn'] = np.log(nasdq_df['CLOSE'] / nasdq_df['CLOSE'].shift(1))
returns_imoex = imoex_df['LogReturn'].dropna() * 100

In [42]:
nasdq_df = pd.read_csv('dataset\\nasdq.csv')
nasdq_df['Date'] = pd.to_datetime(nasdq_df['Date'])
nasdq_df['LogReturn'] = np.log(nasdq_df['Close'] / nasdq_df['Close'].shift(1))
returns_nasdq = nasdq_df['LogReturn'].dropna() * 100

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=['NASDQ', 'IMOEX'])
fig.add_trace(go.Scatter(x=nasdq_df["Date"], y=returns_nasdq, mode='lines', name='NASDQ', line_shape='hv'), row=1, col=1)
fig.add_trace(go.Scatter(x=imoex_df["TRADEDATE"], y=returns_imoex, mode='lines', name='IMOEX', line_shape='hv'), row=2, col=1)
fig.update_xaxes(title_text='день - x', row=2, col=1)
fig.update_yaxes(title_text='индекс - y', row=1, col=1)
fig.update_yaxes(title_text='индекс - y', row=2, col=1)
fig.update_layout(title='Сравнение доходностей NASDQ и IMOEX')
fig.show()

In [ ]:
imoex_train_mask = imoex_df['TRADEDATE'] < '2019-06-06'
imoex_val_mask = (imoex_df['TRADEDATE'] >= '2019-06-06') & (imoex_df['TRADEDATE'] < '2022-02-01')
imoex_test_mask = imoex_df['TRADEDATE'] >= '2022-02-01'

all_info = imoex_train_mask.sum() + imoex_val_mask.sum() + imoex_test_mask.sum()

print(f"Train: {imoex_train_mask.sum()} дней (2010–2019) {imoex_train_mask.sum() / all_info * 100:.2f}%")
print(f"Val:   {imoex_val_mask.sum()} дней (2020–2021) {imoex_val_mask.sum() / all_info * 100:.2f}%")
print(f"Test:  {imoex_test_mask.sum()} дней (2022–2026) {imoex_test_mask.sum() / all_info * 100:.2f}%")


Train: 2364 дней (2010–2019) 57.98%
Val:   672 дней (2020–2021) 16.48%
Test:  1041 дней (2022–2026) 25.53%


In [35]:
nasdq_train_mask = nasdq_df['Date'] < '2019-06-06'
nasdq_val_mask = (nasdq_df['Date'] >= '2019-06-06') & (nasdq_df['Date'] < '2022-02-01')
nasdq_test_mask = nasdq_df['Date'] >= '2022-02-01'

all_info = nasdq_train_mask.sum() + nasdq_val_mask.sum() + nasdq_test_mask.sum()

print(f"Train: {nasdq_train_mask.sum()} дней (2010–2019) {nasdq_train_mask.sum() / all_info * 100:.2f}%")
print(f"Val:   {nasdq_val_mask.sum()} дней (2020–2021) {nasdq_val_mask.sum() / all_info * 100:.2f}%")
print(f"Test:  {nasdq_test_mask.sum()} дней (2022–2026) {nasdq_test_mask.sum() / all_info * 100:.2f}%")

Train: 2490 дней (2010–2019) 63.62%
Val:   702 дней (2020–2021) 17.94%
Test:  722 дней (2022–2026) 18.45%


In [39]:
window = 90  

imoex_predictions = []
imoex_dates = []

for t in range(window, len(returns_imoex)):  

    test_GARCH_data = returns_imoex[t - window : t]

    model = arch_model(test_GARCH_data, vol='Garch', p=1, q=1, dist='Normal')
    result = model.fit(disp='off')  
    
    forecast = result.forecast(horizon=1)
    predicted_var = forecast.variance.values[-1, 0]  # σ²
    
    imoex_predictions.append(predicted_var)
    imoex_dates.append(imoex_df['TRADEDATE'].iloc[t])

imoex_predictions = np.array(imoex_predictions)
imoex_predicted_vol = np.sqrt(imoex_predictions)

In [57]:
# Вычисляем волатильность
vol_simple = returns_imoex.rolling(20).std()

# Берём только валидные значения и соответствующие им даты
valid_mask = vol_simple.notna()
vol_simple_valid = vol_simple[valid_mask]
dates_for_vol = imoex_df['TRADEDATE'].values[1:][valid_mask]  # даты без первого NaN от diff

print(f"Returns: {len(returns_imoex)}")
print(f"Volatility: {len(vol_simple_valid)}")
print(f"Dates for vol: {len(dates_for_vol)}")

fig = go.Figure()

# GARCH predictions (уже со своими датами)
fig.add_trace(go.Scatter(
    x=imoex_dates, 
    y=np.sqrt(imoex_predictions), 
    name='GARCH Vol', 
    line=dict(color='red')
))

# Simple volatility (с выровненными датами)
fig.add_trace(go.Scatter(
    x=dates_for_vol,          # ← правильные даты
    y=vol_simple_valid,       # ← без NaN
    name='Simple Vol (20d)', 
    line=dict(color='blue', dash='dot')
))

fig.update_layout(title='Волатильность: GARCH vs Simple 20d')
fig.show()

Returns: 4076
Volatility: 4057
Dates for vol: 4057


In [ ]:
window = 90  

nasdq_predictions = []
nasdq_dates = []

for t in range(window, len(returns_nasdq)):  

    test_GARCH_data = returns_nasdq[t - window : t]

    model = arch_model(test_GARCH_data, vol='Garch', p=1, q=1, dist='Normal')
    result = model.fit(disp='off')  
    
    forecast = result.forecast(horizon=1)
    predicted_var = forecast.variance.values[-1, 0]  # σ²
    
    nasdq_predictions.append(predicted_var)
    nasdq_dates.append(nasdq_df['Date'].iloc[t])

nasdq_predictions = np.array(nasdq_predictions)
nasdq_predicted_vol = np.sqrt(nasdq_predictions)

In [58]:
# Вычисляем волатильность
vol_simple = returns_nasdq.rolling(20).std()

# Берём только валидные значения и соответствующие им даты
valid_mask = vol_simple.notna()
vol_simple_valid = vol_simple[valid_mask]
dates_for_vol = nasdq_df['Date'].values[1:][valid_mask]  # даты без первого NaN от diff

print(f"Returns: {len(returns_nasdq)}")
print(f"Volatility: {len(vol_simple_valid)}")
print(f"Dates for vol: {len(dates_for_vol)}")

fig = go.Figure()

# GARCH predictions (уже со своими датами)
fig.add_trace(go.Scatter(
    x=nasdq_dates, 
    y=np.sqrt(nasdq_predictions), 
    name='GARCH Vol', 
    line=dict(color='red')
))

# Simple volatility (с выровненными датами)
fig.add_trace(go.Scatter(
    x=dates_for_vol,          # ← правильные даты
    y=vol_simple_valid,       # ← без NaN
    name='Simple Vol (20d)', 
    line=dict(color='blue', dash='dot')
))

fig.update_layout(title='Волатильность: GARCH vs Simple 20d')
fig.show()

Returns: 3913
Volatility: 3894
Dates for vol: 3894


Валидных точек после rolling: 3983


In [135]:
def prepare_data_for_nn(returns, df, WINDOW = 90, data_name="TRADEDATE"):

    rolling_mean = returns.rolling(window=WINDOW).mean()
    raw_var = (returns - rolling_mean) ** 2

    ground_truth_var = raw_var.rolling(window=5).mean()

    valid_idx = ground_truth_var.dropna().index
    print(f"Валидных точек после rolling: {len(valid_idx)}")

    gt_var = ground_truth_var.dropna().values 
    gt_var = np.log1p(gt_var) 
    trade_dates = df.loc[ground_truth_var.dropna().index, data_name].values

    X_windows = []
    y_targets = []
    window_dates = []

    for i in range(WINDOW, len(gt_var)):
        X_windows.append(gt_var[i - WINDOW : i])  # 90 дней σ²
        y_targets.append(gt_var[i])                # σ² на следующий день
        window_dates.append(trade_dates[i])

    X_windows = np.array(X_windows)  # shape: (N, 90)
    y_targets = np.array(y_targets)  # shape: (N,)
    window_dates = np.array(window_dates)

    print(f"X shape: {X_windows.shape}")
    print(f"y shape: {y_targets.shape}")
    print(f"Период: {window_dates[0]} - {window_dates[-1]}")

    window_dates_dt = window_dates.astype('datetime64')

    train_idx = window_dates_dt < np.datetime64('2020-01-01')
    val_idx = (window_dates_dt >= np.datetime64('2020-01-01')) & (window_dates_dt < np.datetime64('2022-01-01'))
    test_idx = window_dates_dt >= np.datetime64('2022-01-01')

    X_train, y_train = X_windows[train_idx], y_targets[train_idx]
    X_val, y_val = X_windows[val_idx], y_targets[val_idx]
    X_test, y_test = X_windows[test_idx], y_targets[test_idx]
    dates_test = window_dates[test_idx] 

    print(f"Train: {X_train.shape[0]} окон")
    print(f"Val:   {X_val.shape[0]} окон")
    print(f"Test:  {X_test.shape[0]} окон")

    train_mean = X_train.mean()
    train_std = X_train.std()

    X_train_norm = (X_train - train_mean) / train_std
    X_val_norm = (X_val - train_mean) / train_std
    X_test_norm = (X_test - train_mean) / train_std

    y_train_norm = (y_train - train_mean) / train_std
    y_val_norm = (y_val - train_mean) / train_std
    y_test_norm = (y_test - train_mean) / train_std

    print(f"Train mean: {train_mean:.6f}, std: {train_std:.6f}")

    return X_train_norm, y_train_norm, X_val_norm, y_val_norm, X_test_norm, y_test_norm, dates_test, train_mean, train_std, window_dates, ground_truth_var

In [146]:
imoex_train_X, imoex_train_y, imoex_val_X, imoex_val_y, imoex_test_X, imoex_test_y, imoex_test_dates, imoex_train_mean, imoex_train_std, imoex_window_dates, imoex_ground_truth_var = prepare_data_for_nn(returns_imoex, imoex_df, WINDOW=90)

Валидных точек после rolling: 3983
X shape: (3893, 90)
y shape: (3893,)
Период: 2010-10-04 - 2026-04-03
Train: 2326 окон
Val:   505 окон
Test:  1062 окон
Train mean: 0.717934, std: 0.481353


In [147]:
nasdq_train_X, nasdq_train_y, nasdq_val_X, nasdq_val_y, nasdq_test_X, nasdq_test_y, nasdq_test_dates, nasdq_train_mean, nasdq_train_std, nasdq_window_dates, nasdq_ground_truth_var = prepare_data_for_nn(returns_nasdq, nasdq_df, WINDOW=90, data_name="Date")

Валидных точек после rolling: 3820
X shape: (3730, 90)
y shape: (3730,)
Период: 2010-09-15T00:00:00.000000000 - 2024-10-25T00:00:00.000000000
Train: 2457 окон
Val:   529 окон
Test:  744 окон
Train mean: 0.879712, std: 0.586086


In [81]:
class VolatilityDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        # X shape: (N, 90) -> (N, 90, 1) для LSTM
        self.X = torch.FloatTensor(X).unsqueeze(-1)  # (N, 90, 1)
        self.y = torch.FloatTensor(y)                 # (N,)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [98]:
Vol_imoex_train_dataset = VolatilityDataset(imoex_train_X, imoex_train_y)
Vol_imoex_val_dataset = VolatilityDataset(imoex_val_X, imoex_val_y)
Vol_imoex_test_dataset = VolatilityDataset(imoex_test_X, imoex_test_y)

BATCH_SIZE = 64

Vol_imoex_train_loader = DataLoader(Vol_imoex_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
Vol_imoex_val_loader = DataLoader(Vol_imoex_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
Vol_imoex_test_loader = DataLoader(Vol_imoex_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"IMOEX Train batches: {len(Vol_imoex_train_loader)}")
print(f"IMOEX Val batches:   {len(Vol_imoex_val_loader)}")
print(f"IMOEX Test batches:  {len(Vol_imoex_test_loader)}")


IMOEX Train batches: 37
IMOEX Val batches:   8
IMOEX Test batches:  17


In [99]:
Vol_nasdq_train_dataset = VolatilityDataset(nasdq_train_X, nasdq_train_y)
Vol_nasdq_val_dataset = VolatilityDataset(nasdq_val_X, nasdq_val_y)
Vol_nasdq_test_dataset = VolatilityDataset(nasdq_test_X, nasdq_test_y)

BATCH_SIZE = 64

Vol_nasdq_train_loader = DataLoader(Vol_nasdq_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
Vol_nasdq_val_loader = DataLoader(Vol_nasdq_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
Vol_nasdq_test_loader = DataLoader(Vol_nasdq_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"NASDQ Train batches: {len(Vol_nasdq_train_loader)}")
print(f"NASDQ Val batches:   {len(Vol_nasdq_val_loader)}")
print(f"NASDQ Test batches:  {len(Vol_nasdq_test_loader)}")


NASDQ Train batches: 39
NASDQ Val batches:   9
NASDQ Test batches:  12


In [100]:
class VolatilityLSTM(nn.Module):
    """
    LSTM модель для предсказания волатильности
    Архитектура из статьи:
    - 3 LSTM слоя с шириной 256
    - Dropout слои между LSTM слоями
    - 2 линейных слоя (fully connected)
    - 1 BatchNorm слой
    - 1 ReLU активация
    """
    def __init__(self, input_size=1, hidden_size=256, num_layers=3, dropout=0.2):
        super(VolatilityLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        self.batch_norm = nn.BatchNorm1d(hidden_size)
        
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 1)
        
        self.dropout = nn.Dropout(dropout)
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        last_output = lstm_out[:, -1, :] 
        normalized = self.batch_norm(last_output)
        
        fc1_out = self.relu(self.fc1(normalized))
        fc1_out = self.dropout(fc1_out)
        
        output = self.fc2(fc1_out)
        
        return output.squeeze(-1)

In [85]:
lstm_model = VolatilityLSTM(hidden_size=256, num_layers=3)
criterion = nn.MSELoss()
optimizer = optim.AdamW(lstm_model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=20, factor=0.5)

lstm_model.to(device)

VolatilityLSTM(
  (lstm): LSTM(1, 256, num_layers=3, batch_first=True, dropout=0.2)
  (batch_norm): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (relu): ReLU()
)

In [86]:
class GatingDataset(Dataset):
    def __init__(self, X, y_real, y_garch):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y_real = torch.FloatTensor(y_real)
        self.y_garch = torch.FloatTensor(y_garch)
    def __len__(self): return len(self.y_real)
    def __getitem__(self, idx): return self.X[idx], self.y_real[idx], self.y_garch[idx]

In [87]:
class FeatureGatingGINN(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.data_lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.data_bn = nn.BatchNorm1d(hidden_size)
        self.garch_proj = nn.Sequential(
            nn.Linear(1, hidden_size // 2), nn.ReLU(), nn.Linear(hidden_size // 2, hidden_size)
        )
        self.lambda_lstm = nn.LSTM(input_size, 32, 2, batch_first=True, bidirectional=True)
        self.lambda_bn = nn.BatchNorm1d(64)
        self.lambda_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1)
        )
    def forward(self, x, garch_signal):
        out, _ = self.data_lstm(x)
        data_feat = self.data_bn(out[:, -1, :])
        garch_feat = self.garch_proj(garch_signal)
        l_out, _ = self.lambda_lstm(x)
        l_feat = self.lambda_bn(l_out[:, -1, :])
        lambda_val = self.lambda_head(l_feat)
        combined = data_feat + lambda_val * garch_feat
        output = self.output_head(combined).squeeze(-1)
        return output, lambda_val.squeeze(-1)

In [103]:
garch_log = np.log1p(imoex_predictions) # логарифм от GARCH предсказаний (σ²)
garch_all_norm = (garch_log - imoex_train_mean) / imoex_train_std  

# А для y используем train_mean/train_std как раньше
imoex_y_train_norm = (imoex_train_y - imoex_train_mean) / imoex_train_std

imoex_garch_train_norm = garch_all_norm[:len(imoex_train_X)]
imoex_garch_val_norm = garch_all_norm[len(imoex_train_X):len(imoex_train_X) + len(imoex_val_X)]
imoex_garch_test_norm = garch_all_norm[len(imoex_train_X) + len(imoex_val_X):]\

imoex_train_dataset = GatingDataset(imoex_train_X, imoex_train_y, imoex_garch_train_norm)
imoex_val_dataset = GatingDataset(imoex_val_X, imoex_val_y, imoex_garch_val_norm)
imoex_test_dataset = GatingDataset(imoex_test_X, imoex_test_y, imoex_garch_test_norm)

BATCH_SIZE = 64

fgginn_imoex_train_loader = DataLoader(imoex_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
fgginn_imoex_val_loader = DataLoader(imoex_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
fgginn_imoex_test_loader = DataLoader(imoex_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"IMOEX Train batches: {len(fgginn_imoex_train_loader)}")
print(f"IMOEX Val batches:   {len(fgginn_imoex_val_loader)}")
print(f"IMOEX Test batches:  {len(fgginn_imoex_test_loader)}")


IMOEX Train batches: 37
IMOEX Val batches:   8
IMOEX Test batches:  17


In [104]:
garch_log = np.log1p(nasdq_predictions) # логарифм от GARCH предсказаний (σ²)
garch_all_norm = (garch_log - nasdq_train_mean) / nasdq_train_std  

# А для y используем train_mean/train_std как раньше
nasdq_y_train_norm = (nasdq_train_y - nasdq_train_mean) / nasdq_train_std

nasdq_garch_train_norm = garch_all_norm[:len(nasdq_train_X)]
nasdq_garch_val_norm = garch_all_norm[len(nasdq_train_X):len(nasdq_train_X) + len(nasdq_val_X)]
nasdq_garch_test_norm = garch_all_norm[len(nasdq_train_X) + len(nasdq_val_X):]\

nasdq_train_dataset = GatingDataset(nasdq_train_X, nasdq_train_y, nasdq_garch_train_norm)
nasdq_val_dataset = GatingDataset(nasdq_val_X, nasdq_val_y, nasdq_garch_val_norm)
nasdq_test_dataset = GatingDataset(nasdq_test_X, nasdq_test_y, nasdq_garch_test_norm)

BATCH_SIZE = 64

fgginn_nasdq_train_loader = DataLoader(nasdq_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
fgginn_nasdq_val_loader = DataLoader(nasdq_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
fgginn_nasdq_test_loader = DataLoader(nasdq_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"NASDQ Train batches: {len(fgginn_nasdq_train_loader)}")
print(f"NASDQ Val batches:   {len(fgginn_nasdq_val_loader)}")
print(f"NASDQ Test batches:  {len(fgginn_nasdq_test_loader)}")


NASDQ Train batches: 39
NASDQ Val batches:   9
NASDQ Test batches:  12


In [128]:
def train_lstm(model, train_loader, val_loader, epochs=300, lr=1e-3, patience=50, model_name='LSTM'):
    """
    Обучение LSTM для прогнозирования волатильности.
    
    Параметры:
    - model: экземпляр VolatilityLSTM
    - train_loader, val_loader: DataLoader'ы
    - epochs: максимальное число эпох
    - lr: learning rate
    - patience: терпение для early stopping
    - model_name: имя модели для вывода
    
    Возвращает: model, history (dict с логами обучения)
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=20, factor=0.5, min_lr=1e-6)
    
    best_val = float('inf')
    best_state = None
    patience_cnt = 0
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_losses.append(loss.item())
        
        avg_train = np.mean(train_losses)
        
        # --- Val ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                val_losses.append(criterion(preds, y_batch).item())
        
        avg_val = np.mean(val_losses)
        scheduler.step(avg_val)
        
        # --- Сохраняем лучшую модель ---
        if avg_val < best_val:
            best_val = avg_val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        
        # --- Логи ---
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        if (epoch + 1) % 25 == 0:
            print(f"[{model_name}] Epoch {epoch+1:3d}/{epochs} | "
                  f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # --- Early stopping ---
        if patience_cnt >= patience:
            print(f"[{model_name}] Early stop at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_state)
    print(f"[{model_name}] Обучение завершено. Best Val Loss: {best_val:.4f}")
    return model, history


def train_fgginn(model, train_loader, val_loader, epochs=300, lr=1e-3, 
                 patience=50, lambda_reg=0.001, model_name='FG-GINN'):
    """
    Обучение Feature Gating GINN.
    
    Параметры:
    - model: экземпляр FeatureGatingGINN
    - train_loader, val_loader: DataLoader'ы для GatingDataset
    - epochs: максимальное число эпох
    - lr: learning rate
    - patience: терпение для early stopping
    - lambda_reg: коэффициент регуляризации λ к 0.5
    - model_name: имя модели для вывода
    
    Возвращает: model, history (dict с логами обучения и λ)
    """
    model = model.to(device)
    mse = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=20, factor=0.5, min_lr=1e-6)
    
    best_val = float('inf')
    best_state = None
    patience_cnt = 0
    
    history = {'train_loss': [], 'val_loss': [], 'train_lambda': [], 'val_lambda': [], 'lr': []}
    
    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_losses = []
        train_lambdas = []
        
        for X_batch, y_real, y_garch in train_loader:
            X_batch = X_batch.to(device)
            y_real = y_real.to(device)
            y_garch = y_garch.to(device)
            
            optimizer.zero_grad()
            preds, lam = model(X_batch, y_garch.unsqueeze(-1))
            
            # Основной loss + регуляризация λ
            loss = mse(preds, y_real) + lambda_reg * ((lam - 0.5) ** 2).mean()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_losses.append(loss.item())
            train_lambdas.append(lam.mean().item())
        
        avg_train = np.mean(train_losses)
        avg_train_lambda = np.mean(train_lambdas)
        
        # --- Val ---
        model.eval()
        val_losses = []
        val_lambdas = []
        
        with torch.no_grad():
            for X_batch, y_real, y_garch in val_loader:
                X_batch = X_batch.to(device)
                y_real = y_real.to(device)
                y_garch = y_garch.to(device)
                
                preds, lam = model(X_batch, y_garch.unsqueeze(-1))
                loss = mse(preds, y_real)
                
                val_losses.append(loss.item())
                val_lambdas.append(lam.mean().item())
        
        avg_val = np.mean(val_losses)
        avg_val_lambda = np.mean(val_lambdas)
        
        scheduler.step(avg_val)
        
        # --- Сохраняем лучшую модель ---
        if avg_val < best_val:
            best_val = avg_val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        
        # --- Логи ---
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['train_lambda'].append(avg_train_lambda)
        history['val_lambda'].append(avg_val_lambda)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        if (epoch + 1) % 25 == 0:
            print(f"[{model_name}] Epoch {epoch+1:3d}/{epochs} | "
                  f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
                  f"λ_tr: {avg_train_lambda:.3f} | λ_val: {avg_val_lambda:.3f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # --- Early stopping ---
        if patience_cnt >= patience:
            print(f"[{model_name}] Early stop at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_state)
    print(f"[{model_name}] Обучение завершено. Best Val Loss: {best_val:.4f}, λ_mean: {avg_val_lambda:.3f}")
    return model, history

def test_model(model, test_loader, train_mean, train_std, model_type='LSTM'):
    model.eval()
    preds, lambdas = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            if model_type == 'LSTM':
                X_batch = batch[0].to(device)
                p = model(X_batch).cpu().numpy()
                preds.extend(p)
            else:  # FG-GINN
                X_batch, _, y_garch = batch
                X_batch = X_batch.to(device)
                y_garch = y_garch.to(device)
                p, l = model(X_batch, y_garch.unsqueeze(-1))
                preds.extend(p.cpu().numpy())
                lambdas.extend(l.cpu().numpy())
    
    preds = np.expm1(np.array(preds) * train_std + train_mean)
    
    if model_type == 'FG-GINN':
        return preds, np.array(lambdas)
    return preds, None

In [ ]:
lstm_model = VolatilityLSTM(hidden_size=256, num_layers=3)
lstm_model, lstm_history = train_lstm(
    lstm_model, 
    Vol_imoex_train_loader, 
    Vol_imoex_val_loader, 
    epochs=300, 
    patience=50,
    model_name='LSTM-IMOEX'
)

fgginn_model = FeatureGatingGINN(hidden_size=128, num_layers=3)
fgginn_model, fgginn_history = train_fgginn(
    fgginn_model,
    fgginn_imoex_train_loader,
    fgginn_imoex_val_loader,
    epochs=300,
    patience=50,
    lambda_reg=0.001,
    model_name='FG-GINN-IMOEX'
)


[FG-GINN-IMOEX] Epoch  25/300 | Train: 0.1958 | Val: 0.2341 | λ_tr: 0.464 | λ_val: 0.456 | LR: 1.00e-03
[FG-GINN-IMOEX] Epoch  50/300 | Train: 0.1756 | Val: 0.1936 | λ_tr: 0.431 | λ_val: 0.403 | LR: 2.50e-04
[FG-GINN-IMOEX] Epoch  75/300 | Train: 0.1648 | Val: 0.2119 | λ_tr: 0.430 | λ_val: 0.421 | LR: 2.50e-04
[FG-GINN-IMOEX] Epoch 100/300 | Train: 0.1625 | Val: 0.1858 | λ_tr: 0.418 | λ_val: 0.408 | LR: 1.25e-04
[FG-GINN-IMOEX] Early stop at epoch 113
[FG-GINN-IMOEX] Обучение завершено. Best Val Loss: 0.1757, λ_mean: 0.425


In [ ]:
lstm_preds, _ = test_model(lstm_model, Vol_imoex_test_loader, 
                            imoex_train_mean, imoex_train_std, 'LSTM')

fgginn_preds, fgginn_lambdas = test_model(fgginn_model, fgginn_imoex_test_loader,
                                           imoex_train_mean, imoex_train_std, 'FG-GINN')

In [151]:
gt_var_imoex = imoex_ground_truth_var.dropna().values  
y_test_real = gt_var_imoex[-imoex_test_X.shape[0]:]

garch_test_preds = imoex_predictions[-imoex_test_X.shape[0]:]

def compute_metrics(y_true, y_pred, model_name):
    """Вычисляет MSE, MAE, R² и выводит результат"""
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MSE': mse, 'MAE': mae, 'R2': r2}

metrics_list = []
metrics_list.append(compute_metrics(y_test_real, garch_test_preds, 'GARCH(1,1)'))
metrics_list.append(compute_metrics(y_test_real, lstm_preds, 'LSTM'))
metrics_list.append(compute_metrics(y_test_real, fgginn_preds, 'FG-GINN'))

df_metrics = pd.DataFrame(metrics_list)
df_metrics = df_metrics.set_index('Model')

print("РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ IMOEX (2022–2026)")

best_r2_model = df_metrics['R2'].idxmax()
print(f"Лучшая модель по R²: {best_r2_model} (R² = {df_metrics.loc[best_r2_model, 'R2']:.4f})")

df_metrics

ValueError: Found input variables with inconsistent numbers of samples: [1062, 744]

In [124]:
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        'IMOEX: Все модели vs Реальность',
        'IMOEX: Ошибки предсказаний (Predicted - Real)',
        'FG-GINN: Адаптивный λ(t)'
    ),
    vertical_spacing=0.1,
    row_heights=[0.4, 0.3, 0.3]
)

# --- Row 1: Все модели ---
fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=y_test_real, name='Realized σ²',
               line=dict(color='black', width=1.5), opacity=0.5),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=garch_test_preds, name='GARCH(1,1)',
               line=dict(color='red', width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=lstm_preds, name='LSTM',
               line=dict(color='orange', width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=fgginn_preds, name='FG-GINN',
               line=dict(color='blue', width=2)),
    row=1, col=1
)

# --- Row 2: Ошибки ---
garch_errors = garch_test_preds - y_test_real
lstm_errors = lstm_preds - y_test_real
fgginn_errors = fgginn_preds - y_test_real

fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=garch_errors, name='GARCH Error',
               line=dict(color='red', width=1), opacity=0.6),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=lstm_errors, name='LSTM Error',
               line=dict(color='orange', width=1), opacity=0.6),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=fgginn_errors, name='FG-GINN Error',
               line=dict(color='blue', width=1.5), opacity=0.8),
    row=2, col=1
)
fig.add_hline(y=0, line=dict(color='gray', dash='dash', width=1), row=2, col=1)

# --- Row 3: Lambda ---
fig.add_trace(
    go.Scatter(x=imoex_test_dates, y=fgginn_lambdas, name='λ(t)',
               line=dict(color='purple', width=1.5),
               fill='tozeroy', fillcolor='rgba(128,0,128,0.15)'),
    row=3, col=1
)
fig.add_hline(y=0.5, line=dict(color='green', dash='dash', width=1.5), 
              annotation_text="Balance λ=0.5", row=3, col=1)

# Зоны интерпретации
fig.add_hrect(y0=0, y1=0.3, fillcolor="red", opacity=0.08, line_width=0, row=3, col=1)
fig.add_hrect(y0=0.7, y1=1.0, fillcolor="blue", opacity=0.08, line_width=0, row=3, col=1)

# Оси
fig.update_yaxes(title_text="σ² variance", row=1, col=1)
fig.update_yaxes(title_text="Error (Pred-Real)", row=2, col=1)
fig.update_yaxes(title_text="λ (GARCH weight)", range=[0, 1], row=3, col=1)
fig.update_xaxes(title_text="Дата", row=3, col=1)

fig.update_layout(
    title=f'IMOEX: сравнение GARCH, LSTM и FG-GINN (лучшая: {best_r2_model})',
    height=1100,
    hovermode='x unified',
    showlegend=True
)
fig.show()

In [130]:
lstm_model = VolatilityLSTM(hidden_size=256, num_layers=3)
lstm_model, lstm_history = train_lstm(
    lstm_model, 
    Vol_nasdq_train_loader, 
    Vol_nasdq_val_loader, 
    epochs=300, 
    patience=50,
    model_name='LSTM-IMOEX'
)

fgginn_model = FeatureGatingGINN(hidden_size=128, num_layers=3)
fgginn_model, fgginn_history = train_fgginn(
    fgginn_model,
    fgginn_nasdq_train_loader,
    fgginn_nasdq_val_loader,
    epochs=300,
    patience=50,
    lambda_reg=0.001,
    model_name='FG-GINN-IMOEX'
)

[LSTM-IMOEX] Epoch  25/300 | Train: 0.1905 | Val: 0.2333 | LR: 1.00e-03
[LSTM-IMOEX] Epoch  50/300 | Train: 0.1766 | Val: 0.2110 | LR: 1.00e-03
[LSTM-IMOEX] Epoch  75/300 | Train: 0.1611 | Val: 0.2208 | LR: 1.00e-03
[LSTM-IMOEX] Epoch 100/300 | Train: 0.1564 | Val: 0.1697 | LR: 5.00e-04
[LSTM-IMOEX] Epoch 125/300 | Train: 0.1362 | Val: 0.1687 | LR: 2.50e-04
[LSTM-IMOEX] Early stop at epoch 138
[LSTM-IMOEX] Обучение завершено. Best Val Loss: 0.1374
[FG-GINN-IMOEX] Epoch  25/300 | Train: 0.2268 | Val: 0.1903 | λ_tr: 0.373 | λ_val: 0.415 | LR: 1.00e-03
[FG-GINN-IMOEX] Epoch  50/300 | Train: 0.1799 | Val: 0.1763 | λ_tr: 0.360 | λ_val: 0.359 | LR: 5.00e-04
[FG-GINN-IMOEX] Epoch  75/300 | Train: 0.1727 | Val: 0.1609 | λ_tr: 0.351 | λ_val: 0.405 | LR: 5.00e-04
[FG-GINN-IMOEX] Epoch 100/300 | Train: 0.1627 | Val: 0.1968 | λ_tr: 0.324 | λ_val: 0.394 | LR: 5.00e-04
[FG-GINN-IMOEX] Epoch 125/300 | Train: 0.1430 | Val: 0.1634 | λ_tr: 0.333 | λ_val: 0.386 | LR: 2.50e-04
[FG-GINN-IMOEX] Early stop a

In [131]:
lstm_preds, _ = test_model(lstm_model, Vol_nasdq_test_loader, 
                            nasdq_train_mean, nasdq_train_std, 'LSTM')

fgginn_preds, fgginn_lambdas = test_model(fgginn_model, fgginn_nasdq_test_loader,
                                           nasdq_train_mean, nasdq_train_std, 'FG-GINN')

In [150]:
gt_var_nasdq = nasdq_ground_truth_var.dropna().values  
y_test_real = gt_var_nasdq[-nasdq_test_X.shape[0]:]

garch_test_preds = nasdq_predictions[-nasdq_test_X.shape[0]:]

def compute_metrics(y_true, y_pred, model_name):
    """Вычисляет MSE, MAE, R² и выводит результат"""
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MSE': mse, 'MAE': mae, 'R2': r2}

metrics_list = []
metrics_list.append(compute_metrics(y_test_real, garch_test_preds, 'GARCH(1,1)'))
metrics_list.append(compute_metrics(y_test_real, lstm_preds, 'LSTM'))
metrics_list.append(compute_metrics(y_test_real, fgginn_preds, 'FG-GINN'))

df_metrics = pd.DataFrame(metrics_list)
df_metrics = df_metrics.set_index('Model')

best_r2_model = df_metrics['R2'].idxmax()
print(f"Лучшая модель по R²: {best_r2_model} (R² = {df_metrics.loc[best_r2_model, 'R2']:.4f})")
df_metrics

Лучшая модель по R²: LSTM (R² = 0.6687)


,MSE,MAE,R2
Model,,,
"GARCH(1,1)",8.852957,1.513274,0.095240
LSTM,3.241250,0.566593,0.668749
FG-GINN,3.400475,0.620245,0.652476


In [133]:
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        'NASDAQ: Все модели vs Реальность',
        'NASDAQ: Ошибки предсказаний (Predicted - Real)',
        'FG-GINN: Адаптивный λ(t)'
    ),
    vertical_spacing=0.1,
    row_heights=[0.4, 0.3, 0.3]
)

# --- Row 1: Все модели ---
fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=y_test_real, name='Realized σ²',
               line=dict(color='black', width=1.5), opacity=0.5),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=garch_test_preds, name='GARCH(1,1)',
               line=dict(color='red', width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=lstm_preds, name='LSTM',
               line=dict(color='orange', width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=fgginn_preds, name='FG-GINN',
               line=dict(color='blue', width=2)),
    row=1, col=1
)

# --- Row 2: Ошибки ---
garch_errors = garch_test_preds - y_test_real
lstm_errors = lstm_preds - y_test_real
fgginn_errors = fgginn_preds - y_test_real

fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=garch_errors, name='GARCH Error',
               line=dict(color='red', width=1), opacity=0.6),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=lstm_errors, name='LSTM Error',
               line=dict(color='orange', width=1), opacity=0.6),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=fgginn_errors, name='FG-GINN Error',
               line=dict(color='blue', width=1.5), opacity=0.8),
    row=2, col=1
)
fig.add_hline(y=0, line=dict(color='gray', dash='dash', width=1), row=2, col=1)

# --- Row 3: Lambda ---
fig.add_trace(
    go.Scatter(x=nasdq_test_dates, y=fgginn_lambdas, name='λ(t)',
               line=dict(color='purple', width=1.5),
               fill='tozeroy', fillcolor='rgba(128,0,128,0.15)'),
    row=3, col=1
)
fig.add_hline(y=0.5, line=dict(color='green', dash='dash', width=1.5), 
              annotation_text="Balance λ=0.5", row=3, col=1)

# Зоны интерпретации
fig.add_hrect(y0=0, y1=0.3, fillcolor="red", opacity=0.08, line_width=0, row=3, col=1)
fig.add_hrect(y0=0.7, y1=1.0, fillcolor="blue", opacity=0.08, line_width=0, row=3, col=1)

# Оси
fig.update_yaxes(title_text="σ² variance", row=1, col=1)
fig.update_yaxes(title_text="Error (Pred-Real)", row=2, col=1)
fig.update_yaxes(title_text="λ (GARCH weight)", range=[0, 1], row=3, col=1)
fig.update_xaxes(title_text="Дата", row=3, col=1)

fig.update_layout(
    title=f'NASDAQ: сравнение GARCH, LSTM и FG-GINN (лучшая: {best_r2_model})',
    height=1100,
    hovermode='x unified',
    showlegend=True
)
fig.show()